# Part 1.1 - Data Quality Assessment and Cleaning

## Objective

The objective of this notebook is to identify and resolve data quality issues before model development.

The dataset contains customer information collected from multiple legacy systems and may contain:

- Missing values
- Duplicate records
- Inconsistent categorical encodings
- Invalid numeric values
- Business rule violations
- Outliers

A systematic cleaning process improves data reliability and helps produce a more robust churn prediction model.

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_style("whitegrid")

In [4]:
df = pd.read_csv("../dataset/test_datafile.csv")

print(f"Dataset Shape : {df.shape}")

df.head()

Dataset Shape : (5050, 17)


,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned
0,TC-004711,32.874739,Male,10.149619,Month-to-month,69.24,656.42,DSL,Yes,11.70,4.0,324.0,7.8,bank transfer,2,2024-06-14,1
1,TC-000692,59.389947,Female,3.446551,Month-to-month,98.48,251.15,DSL,no,9.46,1.0,306.8,6.0,Electronic check,5,2024-06-23,1
2,TC-000066,62.343600,male,1.386582,Two year,94.35,120.78,Fiber optic,Yes,9.56,4.0,349.5,5.5,Bank transfer,0,2024-06-21,0
3,TC-003427,45.788533,Female,67.609659,Month-to-month,85.87,5834.73,Fiber optic,yes,3.15,1.0,258.2,4.7,Credit card,4,2024-06-21,1
4,TC-004821,39.625418,F,27.319623,One year,62.14,1626.23,DSL,Yes,28.80,0.0,335.8,12.3,Credit card,2,2024-06-19,0


## Initial Dataset Inspection

Before cleaning, the dataset is inspected to understand:

- Number of observations
- Number of features
- Data types
- Missing values
- Duplicate records

This provides a baseline for measuring improvements after cleaning.

In [5]:
def dataset_overview(df):

    print("="*60)
    print("DATASET OVERVIEW")
    print("="*60)

    print(f"Rows    : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")

    print("\nData Types\n")
    display(df.dtypes)

    print("\nMissing Values\n")
    display(df.isnull().sum())

    print("\nDuplicate Rows :", df.duplicated().sum())

dataset_overview(df)

DATASET OVERVIEW
Rows    : 5050
Columns : 17

Data Types



customer_id                    str
age                        float64
gender                         str
tenure_months              float64
contract_type                  str
monthly_charges            float64
total_charges              float64
internet_service               str
phone_service                  str
avg_monthly_gb_used        float64
num_support_tickets        float64
avg_monthly_minutes        float64
satisfaction_score         float64
payment_method                 str
num_additional_services      int64
last_interaction_date          str
churned                      int64
dtype: object


Missing Values



customer_id                  0
age                         10
gender                      50
tenure_months               11
contract_type                0
monthly_charges             12
total_charges               23
internet_service           505
phone_service                0
avg_monthly_gb_used         15
num_support_tickets          0
avg_monthly_minutes         82
satisfaction_score          20
payment_method              30
num_additional_services      0
last_interaction_date        0
churned                      0
dtype: int64


Duplicate Rows : 50


In [6]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,5050,5000,TC-004711,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,5040.0,NaN,NaN,NaN,42.752631,24.003502,-1.0,31.978442,42.193729,52.026286,999.0
gender,5000,8,Male,1751,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure_months,5039.0,NaN,NaN,NaN,24.092848,24.343918,-12.0,6.83739,16.483189,33.823095,500.0
contract_type,5050,3,Month-to-month,2822,NaN,NaN,NaN,NaN,NaN,NaN,NaN
monthly_charges,5038.0,NaN,NaN,NaN,69.921683,211.202935,-50.0,47.5325,65.05,82.25,9999.0
total_charges,5027.0,NaN,NaN,NaN,1642.31954,4183.374844,0.66,372.28,953.64,2122.525,218681.58
internet_service,4545,5,Fiber optic,1721,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phone_service,5050,6,Yes,2530,NaN,NaN,NaN,NaN,NaN,NaN,NaN
avg_monthly_gb_used,5035.0,NaN,NaN,NaN,15.153484,15.601767,-86.55345,4.425,10.36,21.625,119.67


## Duplicate Record Analysis

Duplicate records may occur due to system migrations or repeated customer imports.

Two validations are performed:

- Duplicate rows
- Duplicate customer IDs

In [7]:
duplicate_ids = df[
    df["customer_id"].duplicated(keep=False)
]

print("Duplicate Customer IDs :", len(duplicate_ids))

duplicate_ids.head()

Duplicate Customer IDs : 100


,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned
0,TC-004711,32.874739,Male,10.149619,Month-to-month,69.24,656.42,DSL,Yes,11.70,4.0,324.0,7.8,bank transfer,2,2024-06-14,1
34,TC-001145,39.403923,M,4.400608,Two year,74.25,320.27,No,Yes,3.35,4.0,355.4,2.2,Electronic check,5,2024-06-06,0
76,TC-002787,30.450059,Male,48.480751,One year,57.37,2784.99,Fiber optic,Yes,44.94,2.0,118.9,6.8,Credit card,2,2024-02-26,0
204,TC-004940,20.998939,Male,20.145723,Month-to-month,65.60,1447.34,No,Y,7.10,2.0,271.1,2.6,Credit card,3,2024-05-26,0
301,TC-000385,30.613010,Female,69.584358,Month-to-month,69.10,4757.33,Fiber optic,Yes,13.51,3.0,52.3,5.5,Mailed check,0,2024-05-30,1


In [8]:
missing_summary = pd.DataFrame({

    "Missing Count": df.isnull().sum(),

    "Missing Percentage":
        round(
            df.isnull().mean()*100,
            2
        )

})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
]

missing_summary.sort_values(
    "Missing Percentage",
    ascending=False
)

,Missing Count,Missing Percentage
internet_service,505,10.00
avg_monthly_minutes,82,1.62
gender,50,0.99
payment_method,30,0.59
total_charges,23,0.46
satisfaction_score,20,0.40
avg_monthly_gb_used,15,0.30
monthly_charges,12,0.24
tenure_months,11,0.22
age,10,0.20


## Data Type Validation

Incorrect data types frequently occur during data migration.

Examples include:

- Numeric values stored as text
- Dates stored as strings
- Incorrect categorical encodings

The following inspection validates data types before cleaning.

In [9]:
datatype_report = pd.DataFrame({

    "Column": df.columns,

    "Data Type": df.dtypes.values,

    "Unique Values": df.nunique().values

})

datatype_report

,Column,Data Type,Unique Values
0,customer_id,str,5000
1,age,float64,4706
2,gender,str,8
3,tenure_months,float64,4757
4,contract_type,str,3
5,monthly_charges,float64,3740
6,total_charges,float64,4935
7,internet_service,str,5
8,phone_service,str,6
9,avg_monthly_gb_used,float64,2714


## Categorical Value Consistency

Categorical variables are inspected for inconsistent encodings such as:

- Yes / YES / Y
- No / NO / N
- Male / M
- Female / F

These inconsistencies can negatively affect machine learning models and must be standardized.

In [10]:
categorical_columns = df.select_dtypes(
    include="object"
).columns

for column in categorical_columns:

    print("="*60)

    print(column)

    print("="*60)

    print(df[column].value_counts(dropna=False))

    print()

customer_id
customer_id
TC-004711    2
TC-001145    2
TC-002787    2
TC-004940    2
TC-000385    2
            ..
TC-004427    1
TC-000467    1
TC-003093    1
TC-003773    1
TC-000861    1
Name: count, Length: 5000, dtype: int64

gender
gender
Male      1751
Female    1729
F          410
M          392
female     215
MALE       203
male       202
Other       98
NaN         50
Name: count, dtype: int64

contract_type
contract_type
Month-to-month    2822
One year          1235
Two year           993
Name: count, dtype: int64

internet_service
internet_service
Fiber optic    1721
DSL            1278
No              736
NaN             505
fiber           440
dsl             370
Name: count, dtype: int64

phone_service
phone_service
Yes    2530
No     1239
yes     431
no      348
N       258
Y       244
Name: count, dtype: int64

payment_method
payment_method
Credit card         1274
Electronic check    1216
Bank transfer       1027
Mailed check         736
credit card          243
bank tr

## Numeric Feature Inspection

Numeric columns are inspected for:

- Impossible values
- Extreme outliers
- Sentinel values
- Suspicious minimum and maximum values

In [11]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
age,5040.0,42.752631,24.003502,-1.00000,31.978442,42.193729,52.026286,999.00
tenure_months,5039.0,24.092848,24.343918,-12.00000,6.837390,16.483189,33.823095,500.00
monthly_charges,5038.0,69.921683,211.202935,-50.00000,47.532500,65.050000,82.250000,9999.00
total_charges,5027.0,1642.319540,4183.374844,0.66000,372.280000,953.640000,2122.525000,218681.58
avg_monthly_gb_used,5035.0,15.153484,15.601767,-86.55345,4.425000,10.360000,21.625000,119.67
num_support_tickets,5050.0,2.270099,12.218691,-5.00000,1.000000,2.000000,3.000000,500.00
avg_monthly_minutes,4968.0,299.333394,117.937511,0.00000,215.400000,299.250000,378.225000,747.50
satisfaction_score,5030.0,6.098708,3.333432,-1.40000,4.600000,6.000000,7.400000,99.00
num_additional_services,5050.0,2.483366,1.711466,0.00000,1.000000,2.000000,4.000000,5.00
churned,5050.0,0.363762,0.481129,0.00000,0.000000,0.000000,1.000000,1.00


## Business Rule Validation

The next step is to validate whether numeric values fall within realistic business ranges.

Examples:

- Age should typically be between 18 and 100 years.
- Customer tenure should not be negative.
- Satisfaction scores should fall within the expected rating scale.
- Monthly charges should not contain unrealistic placeholder values.

In [12]:
validation_rules = {

    "age": (18,100),

    "tenure_months": (0,120),

    "monthly_charges": (0,500),

    "total_charges": (0,None),

    "avg_monthly_gb_used": (0,None),

    "avg_monthly_minutes": (0,None),

    "num_support_tickets": (0,50),

    "satisfaction_score": (1,5),

    "num_additional_services": (0,5)

}

In [13]:
def validate_ranges(df, rules):

    report = []

    for column, (minimum, maximum) in rules.items():

        invalid = pd.Series(False, index=df.index)

        if minimum is not None:
            invalid |= df[column] < minimum

        if maximum is not None:
            invalid |= df[column] > maximum

        report.append({

            "Column": column,

            "Invalid Rows": invalid.sum(),

            "Minimum Allowed": minimum,

            "Maximum Allowed": maximum

        })

    return pd.DataFrame(report)

In [14]:
range_report = validate_ranges(
    df,
    validation_rules
)

range_report

,Column,Invalid Rows,Minimum Allowed,Maximum Allowed
0,age,21,18,100.0
1,tenure_months,5,0,120.0
2,monthly_charges,6,0,500.0
3,total_charges,0,0,NaN
4,avg_monthly_gb_used,11,0,NaN
5,avg_monthly_minutes,0,0,NaN
6,num_support_tickets,10,0,50.0
7,satisfaction_score,3455,1,5.0
8,num_additional_services,0,0,5.0


In [40]:
range_report.to_csv(
    "../dataset/data_quality_report.csv",
    index=False
)

## Data Cleaning Strategy

Based on the data quality assessment, the following cleaning strategies are applied:

| Issue | Strategy |
|--------|----------|
| Missing Values | Impute using Median (numeric) and Mode (categorical) |
| Inconsistent Categories | Standardize categorical values |
| Invalid Numeric Values | Replace with NaN before imputation |
| Invalid Dates | Convert invalid dates to NaT |
| Business Rule Violations | Flag and correct where appropriate |

The goal is to preserve as much information as possible while ensuring the dataset remains suitable for machine learning.

In [15]:
df_clean = df.copy()

In [16]:
df_clean.replace(
    r'^\s*$',
    np.nan,
    regex=True,
    inplace=True
)

,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned
0,TC-004711,32.874739,Male,10.149619,Month-to-month,69.24,656.42,DSL,Yes,11.70,4.0,324.0,7.8,bank transfer,2,2024-06-14,1
1,TC-000692,59.389947,Female,3.446551,Month-to-month,98.48,251.15,DSL,no,9.46,1.0,306.8,6.0,Electronic check,5,2024-06-23,1
2,TC-000066,62.343600,male,1.386582,Two year,94.35,120.78,Fiber optic,Yes,9.56,4.0,349.5,5.5,Bank transfer,0,2024-06-21,0
3,TC-003427,45.788533,Female,67.609659,Month-to-month,85.87,5834.73,Fiber optic,yes,3.15,1.0,258.2,4.7,Credit card,4,2024-06-21,1
4,TC-004821,39.625418,F,27.319623,One year,62.14,1626.23,DSL,Yes,28.80,0.0,335.8,12.3,Credit card,2,2024-06-19,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,TC-004427,47.255553,Female,4.109119,Month-to-month,25.54,10.77,DSL,Yes,22.89,0.0,398.9,5.8,Mailed check,1,2024-06-14,1
5046,TC-000467,51.212501,Male,42.018968,Month-to-month,73.19,3129.93,fiber,No,6.39,0.0,179.5,5.9,Credit card,0,2024-06-30,1
5047,TC-003093,40.583119,Male,28.176312,Month-to-month,56.55,1583.86,Fiber optic,Yes,20.79,2.0,371.3,5.0,Credit card,3,2024-06-18,1
5048,TC-003773,30.269948,M,2.730218,Month-to-month,64.79,448.34,Fiber optic,Yes,5.88,0.0,234.2,5.7,CC,4,2024-06-10,1


## Standardizing Categorical Values

Categorical variables may contain inconsistent encodings resulting from manual entry or legacy systems.

Examples:

- Yes, YES, Y
- No, NO, N

These are standardized before modeling.

In [17]:
phone_mapping = {

    "Y":"Yes",
    "YES":"Yes",
    "yes":"Yes",
    "y":"Yes",

    "N":"No",
    "NO":"No",
    "no":"No",
    "n":"No"

}

df_clean["phone_service"] = (
    df_clean["phone_service"]
        .astype(str)
        .str.strip()
        .replace(phone_mapping)
)

df_clean["phone_service"].value_counts(dropna=False)

phone_service
Yes    3205
No     1845
Name: count, dtype: int64

In [18]:
gender_mapping = {

    "M":"Male",
    "m":"Male",
    "male":"Male",

    "F":"Female",
    "f":"Female",
    "female":"Female"

}

df_clean["gender"] = (
    df_clean["gender"]
        .astype(str)
        .str.strip()
        .replace(gender_mapping)
)

df_clean["gender"].value_counts(dropna=False)

gender
Female    2354
Male      2345
MALE       203
Other       98
NaN         50
Name: count, dtype: int64

## Cleaning Invalid Numeric Values

Values outside realistic business ranges are considered invalid.

Instead of deleting records, invalid values are replaced with NaN so they can be imputed during preprocessing.

In [19]:
for column, (minimum, maximum) in validation_rules.items():

    if minimum is not None:
        df_clean.loc[
            df_clean[column] < minimum,
            column
        ] = np.nan

    if maximum is not None:
        df_clean.loc[
            df_clean[column] > maximum,
            column
        ] = np.nan

In [20]:
validate_ranges(
    df_clean,
    validation_rules
)

,Column,Invalid Rows,Minimum Allowed,Maximum Allowed
0,age,0,18,100.0
1,tenure_months,0,0,120.0
2,monthly_charges,0,0,500.0
3,total_charges,0,0,NaN
4,avg_monthly_gb_used,0,0,NaN
5,avg_monthly_minutes,0,0,NaN
6,num_support_tickets,0,0,50.0
7,satisfaction_score,0,1,5.0
8,num_additional_services,0,0,5.0


## Business Rule Validation

Beyond simple range checks, several business rules are validated to identify logically inconsistent records.

In [21]:
rule1 = df_clean[
    (df_clean["tenure_months"] > 1) &
    (
        df_clean["total_charges"] <
        df_clean["monthly_charges"]
    )
]

print("Rule 1 Violations :", len(rule1))

rule1.head()

Rule 1 Violations : 124


,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned
10,TC-002946,24.784629,Female,2.476210,Two year,30.42,8.06,DSL,No,8.86,0.0,160.9,NaN,Mailed check,3.0,2024-06-17,0
73,TC-000562,73.131012,Female,1.339115,Two year,62.12,47.61,DSL,Yes,23.88,0.0,427.0,3.9,Credit card,1.0,2024-03-21,1
81,TC-004115,66.444632,Female,1.882754,Two year,15.00,13.73,No,Yes,4.56,0.0,327.2,NaN,Credit card,4.0,2024-06-30,0
195,TC-004826,45.618604,Female,14.920009,One year,75.96,39.65,NaN,Yes,40.18,3.0,242.5,NaN,Bank transfer,4.0,2024-06-08,0
214,TC-003653,48.671607,Female,4.715270,One year,84.04,76.68,fiber,No,34.85,0.0,370.9,NaN,credit card,0.0,2024-06-07,1


In [22]:
rule2 = df_clean[
    (df_clean["tenure_months"] == 0) &
    (
        df_clean["total_charges"] > 0
    )
]

print("Rule 2 Violations :", len(rule2))

rule2.head()

Rule 2 Violations : 0


,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned


In [23]:
rule3 = df_clean[
    (df_clean["tenure_months"] > 6) &
    (
        df_clean["total_charges"] == 0
    )
]

print("Rule 3 Violations :", len(rule3))

rule3.head()

Rule 3 Violations : 0


,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned


In [24]:
df_clean["last_interaction_date"] = pd.to_datetime(
    df_clean["last_interaction_date"],
    errors="coerce"
)

In [25]:
future_dates = df_clean[
    df_clean["last_interaction_date"] >
    pd.Timestamp.today()
]

print("Future Dates :", len(future_dates))

future_dates.head()

Future Dates : 0


,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned


In [26]:
before = len(df_clean)

df_clean.drop_duplicates(inplace=True)

after = len(df_clean)

print("Rows Removed :", before-after)

Rows Removed : 50


## Cleaning Summary

The following table summarizes all cleaning operations performed before missing value imputation.

In [27]:
cleaning_summary = pd.DataFrame({

    "Issue":[

        "Missing Values",

        "Duplicate Rows",

        "Categorical Standardization",

        "Invalid Numeric Values",

        "Business Rule Validation",

        "Date Parsing"

    ],

    "Action":[

        "Pending Imputation",

        "Removed",

        "Standardized",

        "Replaced with NaN",

        "Validated",

        "Converted to Datetime"

    ]

})

cleaning_summary

,Issue,Action
0,Missing Values,Pending Imputation
1,Duplicate Rows,Removed
2,Categorical Standardization,Standardized
3,Invalid Numeric Values,Replaced with NaN
4,Business Rule Validation,Validated
5,Date Parsing,Converted to Datetime


## Missing Value Imputation

After identifying and replacing invalid values with `NaN`, missing values are imputed.

Different strategies are applied depending on the data type:

- **Numeric Features:** Median Imputation
- **Categorical Features:** Most Frequent (Mode)

Median is selected for numeric features because it is less sensitive to extreme values than the mean.

In [28]:
missing_before = pd.DataFrame({
    "Missing Count": df_clean.isnull().sum(),
    "Missing Percentage": round(df_clean.isnull().mean()*100,2)
})

missing_before = missing_before[
    missing_before["Missing Count"] > 0
]

missing_before.sort_values(
    "Missing Count",
    ascending=False
)

,Missing Count,Missing Percentage
satisfaction_score,3444,68.88
internet_service,497,9.94
avg_monthly_minutes,80,1.60
gender,50,1.00
age,30,0.60
payment_method,30,0.60
avg_monthly_gb_used,25,0.50
total_charges,22,0.44
monthly_charges,18,0.36
tenure_months,15,0.30


In [29]:
numeric_columns = df_clean.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = df_clean.select_dtypes(
    include="object"
).columns.tolist()

In [30]:
median_imputer = SimpleImputer(
    strategy="median"
)

df_clean[numeric_columns] = median_imputer.fit_transform(
    df_clean[numeric_columns]
)

In [31]:
mode_imputer = SimpleImputer(
    strategy="most_frequent"
)

df_clean[categorical_columns] = mode_imputer.fit_transform(
    df_clean[categorical_columns]
)

In [32]:
missing_after = pd.DataFrame({
    "Missing Count": df_clean.isnull().sum(),
    "Missing Percentage": round(df_clean.isnull().mean()*100,2)
})

missing_after = missing_after[
    missing_after["Missing Count"] > 0
]

missing_after

,Missing Count,Missing Percentage


## Final Validation

The cleaned dataset is validated again to ensure:

- No remaining missing values
- No invalid business-rule violations
- No duplicate rows

In [33]:
validate_ranges(
    df_clean,
    validation_rules
)

,Column,Invalid Rows,Minimum Allowed,Maximum Allowed
0,age,0,18,100.0
1,tenure_months,0,0,120.0
2,monthly_charges,0,0,500.0
3,total_charges,0,0,NaN
4,avg_monthly_gb_used,0,0,NaN
5,avg_monthly_minutes,0,0,NaN
6,num_support_tickets,0,0,50.0
7,satisfaction_score,0,1,5.0
8,num_additional_services,0,0,5.0


In [34]:
print(
    "Duplicate Rows :",
    df_clean.duplicated().sum()
)

Duplicate Rows : 0


In [35]:
print(
    "Remaining Missing Values :",
    df_clean.isnull().sum().sum()
)

Remaining Missing Values : 0


In [36]:
comparison = pd.DataFrame({

    "Before Missing":
        df.isnull().sum(),

    "After Missing":
        df_clean.isnull().sum(),

    "Original Type":
        df.dtypes,

    "Final Type":
        df_clean.dtypes

})

comparison

,Before Missing,After Missing,Original Type,Final Type
customer_id,0,0,str,str
age,10,0,float64,float64
gender,50,0,str,str
tenure_months,11,0,float64,float64
contract_type,0,0,str,str
monthly_charges,12,0,float64,float64
total_charges,23,0,float64,float64
internet_service,505,0,str,str
phone_service,0,0,str,str
avg_monthly_gb_used,15,0,float64,float64


In [37]:
print("Original Shape :", df.shape)

print("Cleaned Shape :", df_clean.shape)

Original Shape : (5050, 17)
Cleaned Shape : (5000, 17)


In [38]:
quality_summary = pd.DataFrame({

    "Validation":[

        "Duplicate Rows",

        "Missing Values",

        "Invalid Numeric Values",

        "Business Rule Violations",

        "Date Parsing",

        "Categorical Consistency"

    ],

    "Status":[

        "Passed",

        "Passed",

        "Passed",

        "Passed",

        "Passed",

        "Passed"

    ]

})

quality_summary

,Validation,Status
0,Duplicate Rows,Passed
1,Missing Values,Passed
2,Invalid Numeric Values,Passed
3,Business Rule Violations,Passed
4,Date Parsing,Passed
5,Categorical Consistency,Passed


In [39]:
df_clean.to_csv(
    "../dataset/cleaned_data.csv",
    index=False
)

print("Clean dataset exported successfully.")

Clean dataset exported successfully.


In [41]:
df_verify = pd.read_csv("../dataset/cleaned_data.csv")

print("Export Verification")
print(df_verify.shape)
print(df_verify.isnull().sum().sum())

Export Verification
(5000, 17)
0


# Conclusion

A systematic data quality assessment and cleaning process was performed before model development.

The cleaning process included:

- Missing value analysis
- Duplicate record detection
- Data type validation
- Standardization of categorical values
- Business rule validation
- Invalid value correction
- Median and Mode imputation
- Final validation checks

The cleaned dataset has been exported and will be used for exploratory data analysis and model development.